# 04 · Validate — integrate binder + switch, reason about ON/OFF states

**Standard slot:** *validate (in silico).* **For Project 12 this is the core integration step:**
couple the top binders to the switch, **model the integrated construct**, and reason about the
**ON/OFF (two-state) behaviour** — with publication-style figures (D3 part 2). The benchmark here is
**switch architecture** (split-reporter vs LOCKR) and the **affinity-vs-dynamic-range** trade-off.

Two-state AF2 modeling of the switch is the `[extension]`: model the OFF (closed / split-apart) and
ON (open / reconstituted) conformations and derive a relative signal. The `mock` backend gives a
deterministic SYNTHETIC ON/OFF so the reasoning plumbing runs anywhere.

Needs `results/all_ranked.csv` (nb 03), `results/switch_designs.csv` + the binder pools (nb 02).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Integrate the top binders with the switch(es)

Take the top binder survivors (nb 03) and couple each to a switch (nb 02). `integrate_binder_switch()`
assembles the construct and carries the binder metrics; `two_state_readout()` then models the ON
(with analyte) and OFF (no analyte) signals and computes the **dynamic_range = on/off** — the headline
sensor metric. Mock numbers are SYNTHETIC (no real luminescence).

In [ ]:
import pandas as pd, numpy as np, os
import biosensor_tools as bt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
switches = pd.read_csv("results/switch_designs.csv")

# Rebuild lightweight binder objects for the top survivors (mock-safe; sequence carried in the pools).
pools = pd.concat([pd.read_csv("results/bindcraft_designs.csv"),
                   pd.read_csv("results/rfdiffusion_designs.csv")], ignore_index=True)
seq_by_id = pools.set_index("design_id")["sequence"].to_dict()

surv = ranked[ranked["layers_passed"] >= 3].copy()
TOP_N = 10
top_binders = (surv.sort_values(["score"], ascending=False)
                   .groupby("paradigm").head(TOP_N))
print(f"coupling {len(top_binders)} top binder survivors to switches (SYNTHETIC if mock)")

def mk_binder(row):
    b = bt.BinderDesign(design_id=str(row["design_id"]), sequence=str(seq_by_id.get(row["design_id"], "M")),
                        paradigm=str(row["paradigm"]), target="ANALYTE")
    b.plddt=row.get("plddt"); b.pae_interaction=row.get("pae_interaction"); b.scrmsd=row.get("scrmsd")
    b.shape_complementarity=row.get("shape_complementarity"); b.rosetta_dG=row.get("rosetta_dG")
    return b

In [ ]:
# Couple each top binder to one representative switch per architecture and model ON/OFF.
constructs = []
reps = switches.groupby("family").first().reset_index()   # one representative switch per family
for _, brow in top_binders.iterrows():
    binder = mk_binder(brow)
    for _, srow in reps.iterrows():
        sw = bt.SwitchDesign(switch_id=str(srow["switch_id"]), family=str(srow["family"]),
                             reporter=str(srow.get("reporter", "")), scaffold=str(srow.get("scaffold", "")),
                             toggle_score=srow.get("toggle_score"), background_leak=srow.get("background_leak"),
                             synthetic=True)
        c = bt.integrate_binder_switch(binder, sw, tool="mock")
        bt.two_state_readout(c, tool="mock")
        constructs.append(c)

cdf = pd.DataFrame([c.as_row() for c in constructs])
cdf.to_csv("results/constructs.csv", index=False)
print("wrote results/constructs.csv", cdf.shape)
cdf.head(6)[["construct_id", "family", "pae_interaction", "on_signal", "off_signal", "dynamic_range"]]

## 2 · Benchmark: switch architecture (split-reporter vs LOCKR)

Compare the two switch architectures on the **dynamic range** they deliver across the same set of top
binders. A fair comparison couples the *same* binders to each architecture and reports the
*distribution* of dynamic range, not the single best. Mock numbers are SYNTHETIC.

In [ ]:
summary = []
for fam, g in cdf.groupby("family"):
    summary.append(dict(family=fam, n=len(g),
                        median_dynamic_range=round(float(g["dynamic_range"].median()), 2),
                        max_dynamic_range=round(float(g["dynamic_range"].max()), 2),
                        median_off_signal=round(float(g["off_signal"].median()), 2)))
summary = pd.DataFrame(summary)
print("switch-architecture benchmark (SYNTHETIC if mock):")
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.6))
fams = list(cdf["family"].unique())
ax.boxplot([cdf[cdf["family"] == f]["dynamic_range"].dropna() for f in fams])
ax.set_xticks(range(1, len(fams) + 1)); ax.set_xticklabels(fams)
ax.set_ylabel("dynamic range (on/off, fold)"); ax.set_xlabel("switch architecture")
ax.set_title("Dynamic range by switch architecture (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p12_architecture.png", dpi=150); plt.show()
print("saved results/p12_architecture.png")

## 3 · The affinity-vs-dynamic-range trade-off

A real tension in biosensor design: a **too-tight** binder can lock the switch ON regardless of the
analyte (no switching), while a **too-weak** one never forms the ON state. Plot the binder metric
(`pae_interaction`, lower = more confident interface) against the modeled **dynamic range** to see the
trade-off. This is the figure that makes the project a *study*: the best *sensor* is not always the
best *binder*. Mock numbers are SYNTHETIC.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for fam, g in cdf.groupby("family"):
    ax.scatter(g["pae_interaction"], g["dynamic_range"], alpha=0.6, label=fam)
ax.set_xlabel("pae_interaction (Å, lower = more confident binder)")
ax.set_ylabel("dynamic range (on/off, fold)")
ax.set_title("Affinity proxy vs dynamic range (EXAMPLE_DATA if mock)")
ax.legend()
plt.tight_layout(); plt.savefig("results/p12_tradeoff.png", dpi=150); plt.show()
print("saved results/p12_tradeoff.png")
print("Read this as: the strongest binder is not automatically the best SENSOR — tune for dynamic range.")

## 4 · Two-state AF2 modeling of the switch `[extension]`

The rigorous version of ON/OFF: model the construct **without** analyte (OFF: split halves apart /
latch in place) and **with** analyte (ON: halves together / latch displaced), then derive a relative
signal from the predicted state populations + interface confidence. On Colab this uses AF2 multi-state
tricks (templates, state-specific MSAs) or a LOCKR cage+key model (A100). Here we scaffold where it
plugs in — the `mock` ON/OFF above stands in for it.

In [ ]:
# Scaffold: on Colab (A100), replace two_state_readout(tool="mock") with tool="af2":
#   for each construct, model OFF and ON conformations and parse a relative signal proxy.
# Pinned upstream (verify): https://github.com/sokrypton/ColabFold  (two-state via templates/MSAs).
# The output is a MODELED proxy, NOT measured luminescence — the assay (notebook 05) measures it.
print("Two-state AF2 modeling [extension]: switch tool='mock' -> 'af2' in two_state_readout on Colab (A100).")
print("These remain MODELED proxies; the real ON/OFF is the luminescence/FRET dose-response in nb 05.")

## 5 · Select the top integrated constructs

Rank constructs by **dynamic range** first (the sensor metric), with the binder confidence
(`pae_interaction`) as a tie-breaker, and save the shortlist for the validation plan (notebook 05).

In [ ]:
cdf["pae_rank"] = cdf["pae_interaction"].rank(ascending=True)   # lower pae = better binder
top_constructs = cdf.sort_values(["dynamic_range", "pae_rank"],
                                 ascending=[False, True]).head(20)
top_constructs.to_csv("results/top_constructs.csv", index=False)
print("wrote results/top_constructs.csv:", top_constructs.shape)
print("by architecture:", top_constructs.groupby("family").size().to_dict())
top_constructs.head(8)[["construct_id", "family", "pae_interaction", "dynamic_range", "off_signal"]]

## D3 (part 2) checklist
- [ ] Top binders coupled to the switch(es); `results/constructs.csv` with ON/OFF + dynamic range.
- [ ] Switch-architecture benchmark (split-reporter vs LOCKR) figure `results/p12_architecture.png`.
- [ ] Affinity-vs-dynamic-range trade-off figure `results/p12_tradeoff.png` + honest discussion.
- [ ] Two-state AF2 modeling `[extension]` wired up (or its scaffold documented for Colab).
- [ ] `results/top_constructs.csv`: ranked by dynamic range, ready for the validation plan.

**Next:** `05_validation_plan.ipynb` — the luminescence/FRET dose-response + LOD + controls.